# biogas-pvrv-qra — run in Colab

Clones the repository, installs the requirements and runs the dynamic QRA (v1.4: hourly CH₄/CO₂/H₂O/H₂S composition, CO₂ toxicity combined with H₂S). A CPU runtime is enough: about two and a half minutes, plus three for the Bayesian-updating demonstration. Section 3 redraws the figures of the research report with Korean labels.


In [ ]:
REPO = 'https://github.com/drinkgarlic/biogas-pvrv-qra.git'
import os
if not os.path.exists('/content/biogas-pvrv-qra'):
    !git clone -q {REPO} /content/biogas-pvrv-qra
%cd /content/biogas-pvrv-qra
!pip -q install -r requirements.txt
!bash colab/get_external_models.sh

## 1. Run the pipeline

In [ ]:
!python colab/run_dynamic_qra.py --updating

In [ ]:
import pandas as pd
r = pd.read_csv('dynamic_qra/out/risk_summary.csv')
r['CO2_share_%'] = 100 * r['PLL_co2_increment'] / r['IR_total']
r[['case', 'f_on', 'openings', 'IR_total', 'IR_worker', 'PLL_h2so', 'PLL_co2_increment', 'CO2_share_%', 'IR_resp_R5', 'IR_rout', 'ch4_kg']]

In [ ]:
from IPython.display import Image, display
for f in ['met_stability', 'qra_event_tree', 'qra_ir_by_case', 'qra_design_options', 'qra_profiles', 'qra_tornado', 'qra_updating', 'dispersion_check', 'farfield_h2s']:
    p = f'dynamic_qra/fig/{f}.png'
    if os.path.exists(p): display(Image(p, width=900))

## 2. Figures of the research report (Korean labels)

Installs a Korean font and redraws the figures of 부록 Ⅵ and Ⅳ장 into `report_figs/out` (see `report_figs/README.md` for the figure numbers).

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
!python report_figs/figs_appendix6_ko.py
for f in sorted(os.listdir('report_figs/out')):
    display(Image(os.path.join('report_figs/out', f), width=800))

## 3. Download or keep the outputs

In [ ]:
!zip -q -r qra_outputs.zip dynamic_qra/out dynamic_qra/fig report_figs/out
from google.colab import files
files.download('qra_outputs.zip')

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/biogas-pvrv-qra'
for d in ['dynamic_qra/out', 'dynamic_qra/fig', 'report_figs/out']:
    shutil.copytree(d, os.path.join(dst, d), dirs_exist_ok=True)
print('copied to', dst)